# 03 - Exploratory Data Analysis

## RHoMIS Analytics — Section C, Group 7

**Notebook goal**

This notebook does not treat "cleaned" as "complete". It starts with a pre-analysis audit, verifies which populations are valid for each question, and then runs EDA only on defensible analysis populations.

**Decision question**

Which regions and farm profiles are most vulnerable to food insecurity, and which structural factors appear most associated with that vulnerability?


**Verified analysis rules**

- Use `foodshortagetime` as the primary full-dataset food-vulnerability outcome.
- Treat FIES and HFIAS as supporting modules, not full-dataset headline metrics.
- Use indicator-file income fields for income analysis instead of raw local-currency survey columns.
- Prefer standardized physical measures, especially `land_cultivated_ha` and harvest-based productivity.

## 1. Setup

We load the canonical ETL output and configure a plotting style that makes comparison charts readable for both notebook review and later Tableau design decisions.

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import seaborn as sns
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "Notebook 03 requires matplotlib and seaborn. Install them in the notebook environment before running."
    ) from exc

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_theme(style="whitegrid", context="talk")
sns.set_palette("deep")

### 1.1 Load the cleaned analysis inputs

Read the cleaned ETL dataset and merge the household-level indicator file so the analysis can use both the project-ready survey fields and the official household indicators in one dataframe.

In [2]:
data_candidates = [
    Path("../data/processed/rhomis_cleaned.csv.gz"),
    Path("data/processed/rhomis_cleaned.csv.gz"),
]
DATA_PATH = next((path for path in data_candidates if path.exists()), None)
assert DATA_PATH is not None, f"Missing cleaned dataset. Checked: {data_candidates}"

df = pd.read_csv(DATA_PATH, compression="gzip")
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows x {df.shape[1]:,} columns")

indicator_candidates = [
    Path("data/raw/indicator_data.tab"),
    Path("../data/raw/indicator_data.tab"),
]
INDICATOR_PATH = next((path for path in indicator_candidates if path.exists()), None)

assert INDICATOR_PATH is not None, (
    "Missing indicator_data.tab. Place indicator_data.tab in data/raw/ before rerunning this notebook."
)

indicator_usecols = [
    "id_unique",
    "hh_size_members",
    "hh_size_mae",
    "land_cultivated_ha",
    "livestock_tlu",
    "nr_months_food_shortage",
    "hfias_status",
    "fies_score",
    "hdds_good_season",
    "hdds_bad_season",
    "hdds_last_month",
    "crop_income_lcu_per_year",
    "livestock_income_lcu_per_year",
    "off_farm_income_lcu_per_year",
    "total_income_lcu_per_year",
    "currency_conversion_lcu_to_ppp",
    "value_farm_products_consumed_lcu_per_hh_per_year",
    "farm_products_consumed_calories_kcal_per_hh_per_year",
    "proportion_of_value_controlled_female_adult",
    "proportion_of_value_controlled_male_adult",
]
indicators = pd.read_csv(INDICATOR_PATH, sep="\t", usecols=indicator_usecols)
indicators = indicators.rename(
    columns={
        "land_cultivated_ha": "land_cultivated_ha_indicator",
        "fies_score": "fies_score_indicator",
        "hfias_status": "hfias_status_indicator",
    }
)
df = df.merge(indicators, on="id_unique", how="left", validate="one_to_one")
print(f"Merged household indicator data: {indicators.shape[1] - 1} fields")
display(df.head(3))

Loaded cleaned dataset: 54,873 rows x 77 columns
Merged household indicator data: 19 fields


,id_unique,year,gps_lat_rounded,gps_lon_rounded,iso_country_code,country,region,local_currency,respondent_is_head,respondentsex,age_malehead,age_femalehead,education_level,count_people,children_under_4,children_4to10,males11to24,females11to24,males25to50,females25to50,malesover50,femalesover50,land_tenure,landcultivated,unitland,land_ownership,farm_labour,crop_count,crop_name_1,crop_harvest_kg_per_year_1,crop_land_area_1,crop_consumed_prop_1,crop_income_per_year_1,crop_who_control_revenue_1,crop_name_2,crop_harvest_kg_per_year_2,crop_land_area_2,crop_consumed_prop_2,crop_income_per_year_2,crop_who_control_revenue_2,crop_name_3,crop_harvest_kg_per_year_3,crop_land_area_3,crop_consumed_prop_3,crop_income_per_year_3,crop_who_control_revenue_3,crop_name_4,crop_name_5,land_irrigated,livestock_sale_income_1,livestock_meat_who_control_eating_1,livestock_sale_income_2,dairy_products_who_control_eating,offfarm_incomes_any,offfarm_who_control_revenue_1,offfarm_who_control_revenue_2,offfarm_income_proportion,foodshortagetime,fies_1,fies_2,fies_3,fies_4,fies_5,fies_6,fies_7,fies_8,hfias_9,hfias_8,hfias_7,hfias_6,hfias_5,hfias_4,hfias_3,hfias_2,hfias_1,household_size_derived,land_cultivated_ha,currency_conversion_lcu_to_ppp,livestock_tlu,hh_size_members,hh_size_mae,land_cultivated_ha_indicator,nr_months_food_shortage,hfias_status_indicator,fies_score_indicator,hdds_good_season,hdds_bad_season,hdds_last_month,crop_income_lcu_per_year,livestock_income_lcu_per_year,off_farm_income_lcu_per_year,total_income_lcu_per_year,value_farm_products_consumed_lcu_per_hh_per_year,farm_products_consumed_calories_kcal_per_hh_per_year,proportion_of_value_controlled_female_adult,proportion_of_value_controlled_male_adult
0,bf_adn_2019_1_1,2019,11.2,-1.0,bf,burkina_faso,NaN,xof,y,m,38.0,78.0,no_school,7.0,1,1,2,1,1,1,0,0,own_land rent_in_land,5.0,acres,male_adult,household reciprocal,3.0,millet,300.0,underhalf,NaN,0.0,NaN,groundnut,100.0,underhalf,most,0.0,NaN,sesame,NaN,underhalf,little,36000.0,male_adult,NaN,NaN,n,0.0,NaN,7000.0,NaN,y,female_adult,NaN,underhalf,y,y,y,y,y,y,y,n,n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,7,2.023430,205.728551,1.02,7,5.56,2.0,4.0,food_secure,6.0,5.0,2.0,NaN,36000.0,7000.0,10750.0,53750.0,16915.064130,1530900.0,0.321340,0.678660
1,bf_adn_2019_2_1,2019,11.2,-1.0,bf,burkina_faso,NaN,xof,n,f,NaN,30.0,no_school,10.0,1,2,2,0,2,2,0,1,rent_in_land,3.0,acres,NaN,household reciprocal,2.0,millet,1100.0,underhalf,most,0.0,NaN,maize,1500.0,half,most,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,n,0.0,NaN,0.0,NaN,n,NaN,NaN,NaN,y,y,y,y,y,y,y,y,y,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,10,1.214058,205.728551,2.30,10,8.02,1.2,3.0,food_secure,8.0,4.0,2.0,NaN,0.0,4000.0,0.0,4000.0,61109.775830,6743100.0,0.469283,0.530717
2,bf_adn_2019_3_1,2019,11.2,-1.0,bf,burkina_faso,NaN,xof,n,f,NaN,NaN,no_school,6.0,0,1,0,1,1,1,1,1,own_land rent_in_land,2.0,acres,male_adult,household reciprocal,2.0,maize,100.0,underhalf,most,0.0,NaN,groundnut,100.0,half,most,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,n,0.0,NaN,NaN,NaN,n,NaN,NaN,NaN,y,y,y,y,y,y,y,y,n,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6,0.809372,205.728551,0.30,6,4.69,0.8,4.0,food_secure,7.0,4.0,2.0,NaN,0.0,0.0,0.0,0.0,5083.333344,652400.0,0.327869,0.672131


## 2. Pre-Analysis Cross-Check

Before drawing charts, we verify which rows are valid for each analysis question. This prevents misleading comparisons when a survey module only exists for part of the dataset.

In [3]:
fies_cols = [f"fies_{i}" for i in range(1, 9)]
hfias_cols = [f"hfias_{i}" for i in range(1, 10)]
crop_name_cols = [f"crop_name_{i}" for i in range(1, 6)]
harvest_cols = [f"crop_harvest_kg_per_year_{i}" for i in range(1, 4)]
crop_income_cols = [f"crop_income_per_year_{i}" for i in range(1, 4)]
livestock_income_cols = [f"livestock_sale_income_{i}" for i in range(1, 3)]
income_cols = crop_income_cols + livestock_income_cols

def pct(valid_rows: int, total_rows: int) -> float:
    return round((valid_rows / total_rows) * 100, 1)

total_rows = len(df)

### 2.1 Currency audit

The cleaned survey file still contains inconsistent local-currency labels. For that reason, raw local-currency income columns are not used as cross-country dashboard metrics.

In [4]:
currency_label_map = {
    "etb": "birr",
    "ghs": "cedi",
    "inr": "inr",
    "rs": "inr",
    "kes": "kes",
    "ksh": "kes",
    "quetzales": "quetzal",
    "ugx": "ugx",
    "ush": "ugx",
}

scale_variant_labels = {"000_tzs", "1000vnd", "k_vnd", "bif_000"}

df["local_currency_clean"] = df["local_currency"].replace(currency_label_map)
df["currency_scale_variant_flag"] = df["local_currency"].isin(scale_variant_labels)

country_currency_counts = (
    df.dropna(subset=["country", "local_currency_clean"])
      .groupby("country")["local_currency_clean"]
      .nunique()
      .rename("currency_labels_after_safe_harmonization")
)

currency_audit = (
    country_currency_counts.reset_index()
    .sort_values(["currency_labels_after_safe_harmonization", "country"], ascending=[False, True])
)
display(currency_audit.head(15))

currency_examples = (
    df.dropna(subset=["country", "local_currency"])
      .groupby("country")["local_currency"]
      .apply(lambda s: sorted(pd.Series(s.astype(str).unique()).tolist()))
      .reset_index(name="observed_labels")
)
display(currency_examples[currency_examples["observed_labels"].map(len) > 1].head(15))

print(
    "Currency rule for notebook 03: use income fields only within-country and only as supporting context; "
    "do not build pooled cross-country income rankings."
)

,country,currency_labels_after_safe_harmonization
16,malawi,3
28,tanzania,3
30,vietnam,3
31,zambia,3
1,burundi,2
2,cambodia,2
6,drc,2
11,ghana,2
12,guatemala,2
0,burkina_faso,1


,country,observed_labels
1,burundi,"[bif, bif_000]"
2,cambodia,"[k_riel, usd]"
6,drc,"[fc, usd]"
9,ethiopia,"[birr, etb]"
11,ghana,"[cedi, ghs, xof]"
12,guatemala,"[lempiras, quetzal, quetzales]"
14,india,"[inr, rs]"
15,kenya,"[kes, ksh]"
16,malawi,"[kwacha, malawi_kwacha, zambian_kwacha]"
28,tanzania,"[000_tzs, tnz, tzs]"


Currency rule for notebook 03: use income fields only within-country and only as supporting context; do not build pooled cross-country income rankings.


### 2.2 Derived analysis fields

Every derived field below is tied to a clear analysis need:

- `crop_diversity_count`: uses the cleaned `crop_count` field where available, with listed crop names as a fallback
- `crop_harvest_total_kg_observed`: aggregates standardized harvest quantities already available in kg
- `land_productivity_kg_per_ha`: physical productivity proxy, avoiding the currency problem
- `total_income_ppp_per_year`: total income converted with the provided PPP conversion factor
- `income_ppp_per_mae`: PPP income adjusted by male-adult-equivalent household size
- `country_income_rank_pct`: within-country income position that avoids pooled currency misuse
- `fies_yes_count`: indicator-file FIES score where available, with complete module count as fallback
- `hfias_status_indicator`: HFIAS status from the household indicator file

Important note:

We still keep analysis-population checks because indicator fields may have different coverage depending on which modules were used in each survey application.

In [5]:
def count_distinct_non_null(values) -> int:
    cleaned = {
        str(value).strip().lower()
        for value in values
        if pd.notna(value) and str(value).strip() != ""
    }
    return len(cleaned)

hfias_frequency_map = {
    "never": 0,
    "monthly": 1,
    "fewpermonth": 2,
    "weekly": 3,
    "fewperweek": 4,
    "daily": 5,
}

df["food_shortage_flag"] = np.where(
    df["foodshortagetime"].isin(["y", "n"]),
    (df["foodshortagetime"] == "y").astype(int),
    np.nan,
)

crop_name_diversity_count = df[crop_name_cols].apply(count_distinct_non_null, axis=1)
crop_count_numeric = pd.to_numeric(df["crop_count"], errors="coerce")
df["crop_diversity_count"] = crop_count_numeric.combine_first(crop_name_diversity_count).clip(lower=0)

df["crop_harvest_total_kg_observed"] = (
    df[harvest_cols]
    .apply(pd.to_numeric, errors="coerce")
    .where(df[harvest_cols].apply(pd.to_numeric, errors="coerce") > 0)
    .sum(axis=1, min_count=1)
)

df["crop_income_total_lcu_observed"] = df[crop_income_cols].sum(axis=1, min_count=1)
df["livestock_income_total_lcu_observed"] = df[livestock_income_cols].sum(axis=1, min_count=1)
df["farm_cash_income_lcu_observed"] = df[income_cols].sum(axis=1, min_count=1)
df["farm_income_source_count_observed"] = df[income_cols].notna().sum(axis=1)
df["total_income_ppp_per_year"] = np.where(
    (df["total_income_lcu_per_year"].notna()) & (df["currency_conversion_lcu_to_ppp"] > 0),
    df["total_income_lcu_per_year"] / df["currency_conversion_lcu_to_ppp"],
    np.nan,
)
df["income_ppp_per_mae"] = np.where(
    (df["total_income_ppp_per_year"].notna()) & (df["hh_size_mae"] > 0),
    df["total_income_ppp_per_year"] / df["hh_size_mae"],
    np.nan,
)
df["farm_value_consumed_ppp_per_year"] = np.where(
    (df["value_farm_products_consumed_lcu_per_hh_per_year"].notna())
    & (df["currency_conversion_lcu_to_ppp"] > 0),
    df["value_farm_products_consumed_lcu_per_hh_per_year"] / df["currency_conversion_lcu_to_ppp"],
    np.nan,
)
df["has_observed_farm_cash_income"] = np.where(
    df["farm_cash_income_lcu_observed"].notna(),
    (df["farm_cash_income_lcu_observed"] > 0).astype(int),
    np.nan,
)
df["country_income_rank_pct"] = df.groupby("country")["total_income_ppp_per_year"].rank(pct=True)

df["land_productivity_kg_per_ha"] = np.where(
    (df["crop_harvest_total_kg_observed"] > 0) & (df["land_cultivated_ha"] > 0),
    df["crop_harvest_total_kg_observed"] / df["land_cultivated_ha"],
    np.nan,
)

df["fies_complete"] = df[fies_cols].notna().all(axis=1)
fies_count_from_module = np.where(df["fies_complete"], df[fies_cols].eq("y").sum(axis=1), np.nan)
df["fies_yes_count"] = df["fies_score_indicator"].combine_first(pd.Series(fies_count_from_module, index=df.index))

df["hfias_complete"] = df[hfias_cols].notna().all(axis=1)
df["hfias_frequency_score"] = np.where(
    df["hfias_complete"],
    df[hfias_cols].apply(
        lambda row: sum(hfias_frequency_map.get(value, np.nan) for value in row),
        axis=1,
    ),
    np.nan,
)

derived_coverage = pd.DataFrame(
    [
        {"field": "crop_diversity_count", "valid_rows": int(df["crop_diversity_count"].notna().sum())},
        {"field": "crop_harvest_total_kg_observed", "valid_rows": int(df["crop_harvest_total_kg_observed"].notna().sum())},
        {"field": "land_productivity_kg_per_ha", "valid_rows": int(df["land_productivity_kg_per_ha"].notna().sum())},
        {"field": "total_income_ppp_per_year", "valid_rows": int(df["total_income_ppp_per_year"].notna().sum())},
        {"field": "income_ppp_per_mae", "valid_rows": int(df["income_ppp_per_mae"].notna().sum())},
        {"field": "country_income_rank_pct", "valid_rows": int(df["country_income_rank_pct"].notna().sum())},
        {"field": "fies_yes_count", "valid_rows": int(df["fies_yes_count"].notna().sum())},
        {"field": "hfias_status_indicator", "valid_rows": int(df["hfias_status_indicator"].notna().sum())},
        {"field": "hfias_frequency_score", "valid_rows": int(df["hfias_frequency_score"].notna().sum())},
    ]
)
derived_coverage["pct_of_dataset"] = derived_coverage["valid_rows"].map(lambda x: pct(x, total_rows))
display(derived_coverage)

,field,valid_rows,pct_of_dataset
0,crop_diversity_count,54873,100.0
1,crop_harvest_total_kg_observed,49934,91.0
2,land_productivity_kg_per_ha,47803,87.1
3,total_income_ppp_per_year,50964,92.9
4,income_ppp_per_mae,48787,88.9
5,country_income_rank_pct,50964,92.9
6,fies_yes_count,25080,45.7
7,hfias_status_indicator,54873,100.0
8,hfias_frequency_score,6847,12.5


### 2.3 Analysis populations

These populations define what we are actually allowed to say. This prevents generic analysis slippage and ensures every metric is tied to the right sample.

In [6]:
analysis_populations = pd.DataFrame(
    [
        {
            "population": "Full cleaned dataset",
            "valid_rows": total_rows,
            "pct_of_dataset": 100.0,
            "required_fields": "None beyond ETL output",
            "allowed_uses": "Country/year coverage, descriptive structure, crop diversity"
        },
        {
            "population": "Primary food-shortage population",
            "valid_rows": int(df["food_shortage_flag"].notna().sum()),
            "pct_of_dataset": pct(int(df["food_shortage_flag"].notna().sum()), total_rows),
            "required_fields": "foodshortagetime",
            "allowed_uses": "Headline vulnerability metric and profile comparisons"
        },
        {
            "population": "Full FIES module population",
            "valid_rows": int(df["fies_complete"].sum()),
            "pct_of_dataset": pct(int(df["fies_complete"].sum()), total_rows),
            "required_fields": "fies_1 to fies_8",
            "allowed_uses": "Food-security drill-down and validation"
        },
        {
            "population": "Full HFIAS module population",
            "valid_rows": int(df["hfias_complete"].sum()),
            "pct_of_dataset": pct(int(df["hfias_complete"].sum()), total_rows),
            "required_fields": "hfias_1 to hfias_9",
            "allowed_uses": "Smaller-module severity drill-down only"
        },
        {
            "population": "Land productivity population",
            "valid_rows": int(df["land_productivity_kg_per_ha"].notna().sum()),
            "pct_of_dataset": pct(int(df["land_productivity_kg_per_ha"].notna().sum()), total_rows),
            "required_fields": "land_cultivated_ha and crop_harvest_total_kg_observed",
            "allowed_uses": "Productivity profiling and driver analysis"
        },
        {
            "population": "Observed farm cash income population",
            "valid_rows": int(df["farm_cash_income_lcu_observed"].notna().sum()),
            "pct_of_dataset": pct(int(df["farm_cash_income_lcu_observed"].notna().sum()), total_rows),
            "required_fields": "crop_income_per_year_* and livestock_sale_income_*",
            "allowed_uses": "Within-country only; not pooled cross-country ranking"
        },
        {
            "population": "Within-country income rank population",
            "valid_rows": int(df["country_income_rank_pct"].notna().sum()),
            "pct_of_dataset": pct(int(df["country_income_rank_pct"].notna().sum()), total_rows),
            "required_fields": "country and observed farm cash income",
            "allowed_uses": "Robust within-country welfare positioning"
        },
    ]
)
display(analysis_populations)

,population,valid_rows,pct_of_dataset,required_fields,allowed_uses
0,Full cleaned dataset,54873,100.0,None beyond ETL output,"Country/year coverage, descriptive structure, crop diversity"
1,Primary food-shortage population,47399,86.4,foodshortagetime,Headline vulnerability metric and profile comparisons
2,Full FIES module population,25073,45.7,fies_1 to fies_8,Food-security drill-down and validation
3,Full HFIAS module population,6847,12.5,hfias_1 to hfias_9,Smaller-module severity drill-down only
4,Land productivity population,47803,87.1,land_cultivated_ha and crop_harvest_total_kg_observed,Productivity profiling and driver analysis
5,Observed farm cash income population,52344,95.4,crop_income_per_year_* and livestock_sale_income_*,Within-country only; not pooled cross-country ranking
6,Within-country income rank population,50964,92.9,country and observed farm cash income,Robust within-country welfare positioning


### 2.4 Domain-level missingness snapshot

This summary keeps the notebook honest about where data are broad enough for headline claims and where they are only suitable for subpopulation analysis.

In [7]:
missingness_snapshot = pd.DataFrame(
    [
        {"field": "foodshortagetime", "missing_rows": int(df["foodshortagetime"].isna().sum())},
        {"field": "fies_1", "missing_rows": int(df["fies_1"].isna().sum())},
        {"field": "hfias_1", "missing_rows": int(df["hfias_1"].isna().sum())},
        {"field": "count_people", "missing_rows": int(df["count_people"].isna().sum())},
        {"field": "household_size_derived", "missing_rows": int(df["household_size_derived"].isna().sum())},
        {"field": "land_cultivated_ha", "missing_rows": int(df["land_cultivated_ha"].isna().sum())},
        {"field": "local_currency", "missing_rows": int(df["local_currency"].isna().sum())},
        {"field": "crop_income_per_year_1", "missing_rows": int(df["crop_income_per_year_1"].isna().sum())},
        {"field": "land_ownership", "missing_rows": int(df["land_ownership"].isna().sum())},
        {"field": "crop_who_control_revenue_1", "missing_rows": int(df["crop_who_control_revenue_1"].isna().sum())},
    ]
)
missingness_snapshot["missing_pct"] = missingness_snapshot["missing_rows"].map(lambda x: pct(x, total_rows))
display(missingness_snapshot.sort_values("missing_pct", ascending=False))

,field,missing_rows,missing_pct
2,hfias_1,48025,87.5
9,crop_who_control_revenue_1,33169,60.4
1,fies_1,29747,54.2
3,count_people,27111,49.4
8,land_ownership,19805,36.1
6,local_currency,11729,21.4
0,foodshortagetime,7474,13.6
7,crop_income_per_year_1,6847,12.5
5,land_cultivated_ha,3369,6.1
4,household_size_derived,0,0.0


## 3. Headline Outcome: Food Shortage Vulnerability

This is the main analytic spine for both notebook 03 and the future Tableau dashboard. It has the best broad coverage and stays closest to the project problem statement.

In [8]:
food_df = df[df["food_shortage_flag"].notna()].copy()
food_df["food_shortage_flag"] = food_df["food_shortage_flag"].astype(int)

food_by_year = (
    food_df.groupby("year", as_index=False)["food_shortage_flag"]
    .agg(food_shortage_rate="mean", households="count")
)
food_by_year["food_shortage_rate"] *= 100

food_by_country = (
    food_df.groupby("country", as_index=False)["food_shortage_flag"]
    .agg(food_shortage_rate="mean", households="count")
)
food_by_country["food_shortage_rate"] *= 100
food_by_country = food_by_country.sort_values("food_shortage_rate", ascending=False)
food_by_country_robust = food_by_country.query("households >= 500").copy()

display(food_by_year)
display(food_by_country_robust.head(15))

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.lineplot(data=food_by_year, x="year", y="food_shortage_rate", marker="o", ax=axes[0])
axes[0].set_title("Food shortage rate by survey year")
axes[0].set_ylabel("Food shortage rate (%)")
axes[0].set_xlabel("Survey year")

top_countries = food_by_country_robust.head(12)
sns.barplot(
    data=top_countries,
    y="country",
    x="food_shortage_rate",
    ax=axes[1],
    orient="h",
)
axes[1].set_title("Highest food-shortage countries (n >= 500)")
axes[1].set_xlabel("Food shortage rate (%)")
axes[1].set_ylabel("Country")

plt.tight_layout()
plt.show()

,year,food_shortage_rate,households
0,2015,83.883495,515
1,2016,72.595379,1861
2,2017,68.329000,4231
3,2018,71.995144,12355
4,2019,67.242173,8688
5,2020,54.644429,4317
6,2021,75.044826,9481
7,2022,54.916318,5736
8,2023,28.837209,215


,country,food_shortage_rate,households
7,drc,94.549419,1376
25,rwanda,94.323627,2713
4,comoros,90.664273,557
16,malawi,89.473684,627
2,burundi,86.993927,1976
21,niger,85.244338,4195
11,ghana,84.236676,2214
17,mali,80.925986,3067
32,zambia,79.479554,1345
1,burkina_faso,78.969957,6058


## 4. Structural Driver Profile

This section focuses on structural measures that remain comparable across households:

- household size
- cultivated land
- crop diversity
- physical land productivity
- irrigation access
- education
- off-farm participation

In [9]:
structural_summary = food_df[
    [
        "household_size_derived",
        "land_cultivated_ha",
        "crop_diversity_count",
        "land_productivity_kg_per_ha",
    ]
].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T

display(structural_summary)

plot_df = food_df.copy()
plot_df["log_land_cultivated_ha"] = np.log10(plot_df["land_cultivated_ha"].where(plot_df["land_cultivated_ha"] > 0))
plot_df["log_land_productivity_kg_per_ha"] = np.log10(
    plot_df["land_productivity_kg_per_ha"].where(plot_df["land_productivity_kg_per_ha"] > 0)
)

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
sns.histplot(data=plot_df, x="household_size_derived", bins=30, ax=axes[0, 0])
axes[0, 0].set_title("Household size distribution")

sns.histplot(data=plot_df.dropna(subset=["log_land_cultivated_ha"]), x="log_land_cultivated_ha", bins=30, ax=axes[0, 1])
axes[0, 1].set_title("Cultivated land distribution (log10 hectares)")

sns.countplot(data=plot_df, x="crop_diversity_count", ax=axes[1, 0])
axes[1, 0].set_title("Crop diversity count distribution")

sns.histplot(
    data=plot_df.dropna(subset=["log_land_productivity_kg_per_ha"]),
    x="log_land_productivity_kg_per_ha",
    bins=30,
    ax=axes[1, 1],
)
axes[1, 1].set_title("Land productivity distribution (log10 kg per ha)")

plt.tight_layout()
plt.show()

,count,mean,std,min,50%,75%,90%,95%,99%,max
household_size_derived,47399.0,7.496297,6.798687e+00,0.00,6.000000,9.000000,14.000000,18.000000,33.0,3.690000e+02
land_cultivated_ha,45053.0,5.492230,1.025164e+02,0.00,1.618744,4.000000,7.000000,11.000000,30.0,1.618744e+04
crop_diversity_count,47399.0,2.333952,1.276352e+00,0.00,2.000000,3.000000,4.000000,5.000000,5.0,1.000000e+01
land_productivity_kg_per_ha,42919.0,66205.052931,2.899725e+06,0.01,593.052391,1665.244399,4714.285714,10666.666667,75000.0,3.916667e+08


### 4.1 Compare the strongest structural profile splits

This block turns the main structural drivers into directly comparable food-shortage rates so the notebook can show which household profiles are most exposed.

In [10]:
profile_rates = pd.DataFrame(
    {
        "dimension": [
            "Irrigated = no",
            "Irrigated = yes",
            "Off-farm income = no",
            "Off-farm income = yes",
            "Respondent sex = female",
            "Respondent sex = male",
        ],
        "food_shortage_rate_pct": [
            round(food_df.loc[food_df["land_irrigated"] == "n", "food_shortage_flag"].mean() * 100, 1),
            round(food_df.loc[food_df["land_irrigated"] == "y", "food_shortage_flag"].mean() * 100, 1),
            round(food_df.loc[food_df["offfarm_incomes_any"] == "n", "food_shortage_flag"].mean() * 100, 1),
            round(food_df.loc[food_df["offfarm_incomes_any"] == "y", "food_shortage_flag"].mean() * 100, 1),
            round(food_df.loc[food_df["respondentsex"] == "f", "food_shortage_flag"].mean() * 100, 1),
            round(food_df.loc[food_df["respondentsex"] == "m", "food_shortage_flag"].mean() * 100, 1),
        ],
    }
)
display(profile_rates)

education_rates = (
    food_df.groupby("education_level", as_index=False)["food_shortage_flag"]
    .agg(food_shortage_rate="mean", households="count")
    .query("households >= 200")
    .sort_values("households", ascending=False)
    .head(12)
)
education_rates["food_shortage_rate"] *= 100
display(education_rates)

prod_df = food_df.dropna(subset=["land_productivity_kg_per_ha"]).copy()
prod_df["productivity_quartile"] = pd.qcut(
    prod_df["land_productivity_kg_per_ha"],
    q=4,
    labels=["Q1 lowest", "Q2", "Q3", "Q4 highest"],
    duplicates="drop",
)
productivity_rates = (
    prod_df.groupby("productivity_quartile", as_index=False)["food_shortage_flag"]
    .agg(food_shortage_rate="mean", households="count")
)
productivity_rates["food_shortage_rate"] *= 100
display(productivity_rates)

income_rank_df = food_df.dropna(subset=["country_income_rank_pct"]).copy()
income_rank_df["income_quintile_within_country"] = pd.qcut(
    income_rank_df["country_income_rank_pct"],
    5,
    labels=["Q1 lowest", "Q2", "Q3", "Q4", "Q5 highest"],
)
income_rank_rates = (
    income_rank_df.groupby("income_quintile_within_country", as_index=False)["food_shortage_flag"]
    .agg(food_shortage_rate="mean", households="count")
)
income_rank_rates["food_shortage_rate"] *= 100
display(income_rank_rates)

fig, axes = plt.subplots(2, 2, figsize=(20, 14))

sns.barplot(data=profile_rates, y="dimension", x="food_shortage_rate_pct", ax=axes[0, 0], orient="h")
axes[0, 0].set_title("Food shortage rate across core profile flags")
axes[0, 0].set_xlabel("Food shortage rate (%)")
axes[0, 0].set_ylabel("")

sns.barplot(data=education_rates, y="education_level", x="food_shortage_rate", ax=axes[0, 1], orient="h")
axes[0, 1].set_title("Food shortage rate by education level")
axes[0, 1].set_xlabel("Food shortage rate (%)")
axes[0, 1].set_ylabel("")

sns.barplot(data=productivity_rates, x="productivity_quartile", y="food_shortage_rate", ax=axes[1, 0])
axes[1, 0].set_title("Food shortage rate by land productivity quartile")
axes[1, 0].set_xlabel("Productivity quartile")
axes[1, 0].set_ylabel("Food shortage rate (%)")

sns.barplot(data=income_rank_rates, x="income_quintile_within_country", y="food_shortage_rate", ax=axes[1, 1])
axes[1, 1].set_title("Food shortage rate by within-country income quintile")
axes[1, 1].set_xlabel("Income quintile within country")
axes[1, 1].set_ylabel("Food shortage rate (%)")

plt.tight_layout()
plt.show()

,dimension,food_shortage_rate_pct
0,Irrigated = no,71.5
1,Irrigated = yes,55.2
2,Off-farm income = no,70.0
3,Off-farm income = yes,64.6
4,Respondent sex = female,72.0
5,Respondent sex = male,64.4


,education_level,food_shortage_rate,households
31,no_school,79.002625,14097
35,primary,69.188053,13326
39,secondary,67.427098,5041
18,adult_education,68.051656,1781
26,koranic,79.873817,1585
34,postsecondary,47.856155,1446
23,illiterate,66.920732,656
37,primary_2,40.207972,577
40,secondary_1,19.259259,540
29,lower_secondary,50.361446,415


,productivity_quartile,food_shortage_rate,households
0,Q1 lowest,82.729469,10764
1,Q2,74.948723,10726
2,Q3,62.043182,10699
3,Q4 highest,53.643989,10730


,income_quintile_within_country,food_shortage_rate,households
0,Q1 lowest,75.254349,9141
1,Q2,75.158609,9142
2,Q3,68.285746,9141
3,Q4,64.332604,9140
4,Q5 highest,55.387813,9141


## 5. Food-Security Module Validation

The goal here is not to replace the headline food-shortage metric. It is to test whether the module-based food security measures tell a compatible story on the rows where they exist.

In [11]:
fies_df = df[df["fies_yes_count"].notna()].copy()
fies_summary = (
    fies_df.groupby("fies_yes_count", as_index=False)
    .agg(households=("id_unique", "count"))
)
display(fies_summary)

fies_food_link = (
    fies_df[fies_df["food_shortage_flag"].notna()]
    .groupby("fies_yes_count", as_index=False)["food_shortage_flag"]
    .agg(food_shortage_rate="mean", households="count")
)
fies_food_link["food_shortage_rate"] *= 100
display(fies_food_link)

fig, ax = plt.subplots(figsize=(10, 6))
sns.lineplot(data=fies_food_link, x="fies_yes_count", y="food_shortage_rate", marker="o", ax=ax)
ax.set_title("Food shortage rate rises sharply with FIES yes-count")
ax.set_xlabel("FIES yes-count (complete module only)")
ax.set_ylabel("Food shortage rate (%)")
plt.tight_layout()
plt.show()

,fies_yes_count,households
0,0.0,6206
1,1.0,2115
2,2.0,1887
3,3.0,2001
4,4.0,2263
5,5.0,2176
6,6.0,1919
7,7.0,3001
8,8.0,3512


,fies_yes_count,food_shortage_rate,households
0,0.0,24.776952,5380
1,1.0,53.984451,2058
2,2.0,67.811159,1864
3,3.0,83.671400,1972
4,4.0,90.965732,2247
5,5.0,91.489362,2162
6,6.0,95.395081,1911
7,7.0,97.560976,2993
8,8.0,96.837607,3510


### 5.1 Validate the headline outcome with HFIAS severity

HFIAS covers a smaller subset than the headline food-shortage variable, so this section is used as a supporting validation layer rather than a full-dataset KPI.

In [12]:
hfias_df = df[df["hfias_frequency_score"].notna()].copy()
hfias_food_link = (
    hfias_df[hfias_df["food_shortage_flag"].notna()]
    .groupby("food_shortage_flag")["hfias_frequency_score"]
    .describe()[["count", "mean", "50%", "75%"]]
    .rename(index={0.0: "No food shortage", 1.0: "Food shortage"})
)
display(hfias_food_link)

fig, ax = plt.subplots(figsize=(10, 6))
sns.boxplot(
    data=hfias_df[hfias_df["food_shortage_flag"].notna()],
    x="food_shortage_flag",
    y="hfias_frequency_score",
    ax=ax,
)
ax.set_title("HFIAS severity proxy by food shortage status")
ax.set_xlabel("Food shortage flag (0 = no, 1 = yes)")
ax.set_ylabel("HFIAS frequency score")
plt.tight_layout()
plt.show()

,count,mean,50%,75%
food_shortage_flag,,,,
No food shortage,3457.0,0.484813,0.0,0.0
Food shortage,2454.0,3.653627,0.0,3.0


## 6. Gender and Resource-Control Drill-Down

The cleaned dataset preserves decision-making fields, but their coverage is too sparse for headline cross-country use. In this notebook, we audit them as possible drill-down dimensions rather than forcing them into the main dashboard narrative.

In [13]:
gender_fields = [
    "land_ownership",
    "crop_who_control_revenue_1",
    "offfarm_who_control_revenue_1",
    "livestock_meat_who_control_eating_1",
    "dairy_products_who_control_eating",
]

gender_coverage = pd.DataFrame(
    [
        {
            "field": field,
            "valid_rows": int(df[field].notna().sum()),
            "pct_of_dataset": pct(int(df[field].notna().sum()), total_rows),
            "top_categories": ", ".join(df[field].dropna().astype(str).value_counts().head(3).index.tolist()),
        }
        for field in gender_fields
    ]
)
display(gender_coverage)

,field,valid_rows,pct_of_dataset,top_categories
0,land_ownership,35068,63.9,"male_adult, male_adult female_adult, female_adult"
1,crop_who_control_revenue_1,21704,39.6,"male_adult female_adult, male_adult, female_adult"
2,offfarm_who_control_revenue_1,19275,35.1,"male_adult, male_adult female_adult, female_adult"
3,livestock_meat_who_control_eating_1,6744,12.3,"male_adult female_adult, male_adult, female_adult"
4,dairy_products_who_control_eating,1289,2.3,"female_adult, male_adult female_adult, male_adult female_adult male_youth female_youth"


## 7. Candidate Dashboard Metrics for Handoff

This is not the final dashboard metric framework. It is an analysis handoff showing which metrics are safe enough for final dashboard design to consider.

In [14]:
tableau_kpis = pd.DataFrame(
    [
        {
            "field_name": "food_shortage_rate",
            "definition": "Mean of food_shortage_flag",
            "scope": "Rows with foodshortagetime in {y, n}",
            "dashboard_role": "Candidate headline metric",
            "status": "Safe candidate for final dashboard design",
        },
        {
            "field_name": "fies_yes_count",
            "definition": "Count of yes responses across 8 FIES items",
            "scope": "Complete FIES module only",
            "dashboard_role": "Supporting food-security drill-down",
            "status": "Safe supporting candidate",
        },
        {
            "field_name": "hfias_frequency_score",
            "definition": "Ordered frequency proxy across 9 HFIAS items",
            "scope": "Complete HFIAS module only",
            "dashboard_role": "Supporting food-security drill-down",
            "status": "Use with sample-size caution",
        },
        {
            "field_name": "crop_diversity_count",
            "definition": "Cleaned crop_count where available; listed crop names as fallback",
            "scope": "Full dataset",
            "dashboard_role": "Profile dimension and comparison metric",
            "status": "Safe candidate for segmentation",
        },
        {
            "field_name": "land_productivity_kg_per_ha",
            "definition": "Observed total crop harvest kg divided by cultivated hectares",
            "scope": "Rows with valid positive harvest and land",
            "dashboard_role": "Structural driver metric",
            "status": "Safe candidate for driver analysis",
        },
        {
            "field_name": "country_income_rank_pct",
            "definition": "Percentile rank of PPP-normalized total income within each country",
            "scope": "Rows with valid income and PPP conversion factor",
            "dashboard_role": "Supporting welfare-position metric",
            "status": "Safe candidate for final dashboard design",
        },
        {
            "field_name": "income_ppp_per_mae",
            "definition": "Total income converted with PPP factor and divided by household MAE",
            "scope": "Rows with valid income, PPP conversion, and MAE household size",
            "dashboard_role": "Secondary income-depth metric",
            "status": "Use medians/log scales because income is highly skewed",
        },
    ]
)
display(tableau_kpis)

,field_name,definition,scope,dashboard_role,status
0,food_shortage_rate,Mean of food_shortage_flag,"Rows with foodshortagetime in {y, n}",Candidate headline metric,Safe candidate for final dashboard design
1,fies_yes_count,Count of yes responses across 8 FIES items,Complete FIES module only,Supporting food-security drill-down,Safe supporting candidate
2,hfias_frequency_score,Ordered frequency proxy across 9 HFIAS items,Complete HFIAS module only,Supporting food-security drill-down,Use with sample-size caution
3,crop_diversity_count,Cleaned crop_count where available; listed crop names as fallback,Full dataset,Profile dimension and comparison metric,Safe candidate for segmentation
4,land_productivity_kg_per_ha,Observed total crop harvest kg divided by cultivated hectares,Rows with valid positive harvest and land,Structural driver metric,Safe candidate for driver analysis
5,country_income_rank_pct,Percentile rank of PPP-normalized total income within each country,Rows with valid income and PPP conversion factor,Supporting welfare-position metric,Safe candidate for final dashboard design
6,income_ppp_per_mae,Total income converted with PPP factor and divided by household MAE,"Rows with valid income, PPP conversion, and MAE household size",Secondary income-depth metric,Use medians/log scales because income is highly skewed


## 8. Verified Notebook Takeaways

This final section prints decision-language takeaways directly from the computed summaries. These statements are designed to guide the Tableau design and the statistical tests in notebook 04.

In [15]:
top_country = food_by_country.query("households >= 500").iloc[0]
irr_gap = (
    food_df.loc[food_df["land_irrigated"] == "n", "food_shortage_flag"].mean()
    - food_df.loc[food_df["land_irrigated"] == "y", "food_shortage_flag"].mean()
) * 100
prod_gap = (
    productivity_rates.loc[productivity_rates["productivity_quartile"] == "Q1 lowest", "food_shortage_rate"].iloc[0]
    - productivity_rates.loc[productivity_rates["productivity_quartile"] == "Q4 highest", "food_shortage_rate"].iloc[0]
)
income_gap = (
    income_rank_rates.loc[income_rank_rates["income_quintile_within_country"] == "Q1 lowest", "food_shortage_rate"].iloc[0]
    - income_rank_rates.loc[income_rank_rates["income_quintile_within_country"] == "Q5 highest", "food_shortage_rate"].iloc[0]
)
fies_gap = (
    fies_food_link.loc[fies_food_link["fies_yes_count"] == 8, "food_shortage_rate"].iloc[0]
    - fies_food_link.loc[fies_food_link["fies_yes_count"] == 0, "food_shortage_rate"].iloc[0]
)

findings = [
    f"Headline metric candidate: foodshortagetime remains the best full-dataset vulnerability measure with {int(food_df.shape[0]):,} valid households.",
    f"Highest-risk country among samples with at least 500 households: {top_country['country']} at {top_country['food_shortage_rate']:.1f}% food shortage.",
    f"Irrigation is a major decision variable: households without irrigation show a food-shortage rate {irr_gap:.1f} percentage points higher than irrigated households.",
    f"Physical land productivity matters strongly: the lowest productivity quartile has a food-shortage rate {prod_gap:.1f} points above the highest quartile.",
    f"PPP-normalized income position is useful: the lowest income quintile sits {income_gap:.1f} points above the highest on food shortage.",
    f"FIES validates the headline story: moving from 0 to 8 yes-responses increases observed food-shortage rates by {fies_gap:.1f} percentage points in the overlapping population.",
    "The analysis uses the provided PPP conversion factor rather than a single 2026 spot-USD conversion.",
]

for idx, finding in enumerate(findings, start=1):
    print(f"{idx}. {finding}")

1. Headline metric candidate: foodshortagetime remains the best full-dataset vulnerability measure with 47,399 valid households.
2. Highest-risk country among samples with at least 500 households: drc at 94.5% food shortage.
3. Irrigation is a major decision variable: households without irrigation show a food-shortage rate 16.2 percentage points higher than irrigated households.
4. Physical land productivity matters strongly: the lowest productivity quartile has a food-shortage rate 29.1 points above the highest quartile.
5. PPP-normalized income position is useful: the lowest income quintile sits 19.9 points above the highest on food shortage.
6. FIES validates the headline story: moving from 0 to 8 yes-responses increases observed food-shortage rates by 72.1 percentage points in the overlapping population.
7. The analysis uses the provided PPP conversion factor rather than a single 2026 spot-USD conversion.
